## Purpose 

This script creates a model for predicting water temperature in Shadow Mountain Reservoir using a random test set of two years of data. Directions for creating the virtual environment are includeed in the README.md file.

### Import Modules

In [19]:
#high level modules
import os
import imp
import pandas as pd

# ml/ai modules
import tensorflow as tf

### Custom Modules

In [20]:
# import custom modules
this_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/operational_model/"
imp.load_source("settings",os.path.join(this_dir,"settings.py"))
from settings import settings
imp.load_source("architecture", os.path.join(this_dir, "architecture.py"))
from architecture import build_model, compile_model
imp.load_source("universals", os.path.join(this_dir, "universal_functions.py"))
from universals import save_to_pickle, twotemp_labels_features

# point to data directory
data_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/operational_model/data/"

### Import train/val sets

Import and format training and validation arrays for use in model training

In [21]:
all_files = pd.Series(os.listdir(data_dir))
test = all_files[all_files.str.contains('test')]
val = all_files[all_files.str.contains('validation')]
train = all_files[all_files.str.contains('training')]

# these files end up in no particular order, so we need to sort them
validation_files = val.sort_values()
training_files = train.sort_values()

# function to load files
def load_data(file):
    return pd.read_csv(os.path.join(data_dir, file), sep=',')

val1 = load_data(validation_files.values[0])
train1 = load_data(training_files.values[0])

val2 = load_data(validation_files.values[1])
train2 = load_data(training_files.values[1])

val3 = load_data(validation_files.values[2])
train3 = load_data(training_files.values[2])

val4 = load_data(validation_files.values[3])
train4 = load_data(training_files.values[3])


In [22]:
val1.columns

Index(['date', 'mean_1m_temp_degC', 'mean_0_5m_temp_degC',
       'mean_1m_temp_degC_m1', 'mean_0_5m_temp_degC_m1',
       'total_solar_radiation', 'total_solar_radiation_m1',
       'total_solar_radiation_m2', 'mean_air_temp', 'min_air_temp',
       'max_air_temp', 'mean_air_temp_m1', 'min_air_temp_m1',
       'max_air_temp_m1', 'mean_rel_hum_m1', 'mean_rel_hum_m2', 'pump_cfs_m1',
       'pump_cfs_m2', 'pump_cfs_m3', 'mean_wind', 'max_wind', 'mean_wind_m1',
       'max_wind_m1', 'nf_cfs_m1', 'nf_cfs_m2', 'nf_cfs_m3', 'nf_cfs_m4',
       'chipmunk_cfs_m1', 'chipmunk_cfs_m2', 'chipmunk_cfs_m3',
       'chipmunk_cfs_m4'],
      dtype='object')

Using the function twotemp_labels_features, we can create ML-ready features and labels for the training and validation sets.

In [23]:
features1, labels_1, val_features1, val_labels_1 = twotemp_labels_features(train1, val1)
features2, labels_2, val_features2, val_labels_2 = twotemp_labels_features(train2, val2)
features3, labels_3, val_features3, val_labels_3 = twotemp_labels_features(train3, val3)
features4, labels_4, val_features4, val_labels_4 = twotemp_labels_features(train4, val4)

### Compile and train models

Look at available settings for the model.

In [24]:
settings.keys()

dict_keys(['overfit', 'simple', 'three_ten', 'three_ten_larger_eta', 'five_ten', 'five_ten_larger_eta'])

make sure the setting is working correctly

In [25]:
model_setting = "five_ten"
print(settings[model_setting]["random_seed"])

57


In [26]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(settings[model_setting]["random_seed"])

# define the early stopping callback
early_stopping_callback = tf.keras.callbacks.EarlyStopping(
  monitor="val_loss", 
  patience=settings[model_setting]["patience"], 
  restore_best_weights=True, 
  mode="auto"
)

## TS cross 1
model_1 = build_model(
  features1, 
  labels_1, 
  settings[model_setting])

model_1 = compile_model(
  model_1, 
  settings[model_setting])

# train the model via model.fit
history_1 = model_1.fit(
  features1, 
  labels_1, 
  epochs=settings[model_setting]["max_epochs"],
  batch_size=settings[model_setting]["batch_size"],
  shuffle=True,
  validation_data=[val_features1, val_labels_1],
  callbacks=[early_stopping_callback],
  verbose=1,
)

## TS cross 2
model_2 = build_model(
  features2,
  labels_2, 
  settings[model_setting])
model_2 = compile_model(model_2, settings[model_setting])

# train the model via model.fit
history_2 = model_2.fit(
  features2,
  labels_2,
  epochs=settings[model_setting]["max_epochs"],
  batch_size=settings[model_setting]["batch_size"],
  shuffle=True,
  validation_data=[val_features2, val_labels_2],
  callbacks=[early_stopping_callback],
  verbose=1,
)

## TS cross 3

model_3 = build_model(
  features3,
  labels_3,
  settings[model_setting])
model_3 = compile_model(model_3, settings[model_setting])

# train the model via model.fit
history_3 = model_3.fit(
  features3,
  labels_3,
  epochs=settings[model_setting]["max_epochs"],
  batch_size=settings[model_setting]["batch_size"],
  shuffle=True,
  validation_data=[val_features3, val_labels_3],
  callbacks=[early_stopping_callback],
  verbose=1,
)

## TS cross 4

model_4 = build_model(
  features4,
  labels_4,
  settings[model_setting])
model_4 = compile_model(model_4, settings[model_setting])

# train the model via model.fit
history_4 = model_4.fit(
  features4,
  labels_4,
  epochs=settings[model_setting]["max_epochs"],
  batch_size=settings[model_setting]["batch_size"],
  shuffle=True,
  validation_data=[val_features4, val_labels_4],
  callbacks=[early_stopping_callback],
  verbose=1,
)


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 28)]              0         
                                                                 
 dropout (Dropout)           (None, 28)                0         
                                                                 
 dense (Dense)               (None, 10)                290       
                                                                 
 dense_1 (Dense)             (None, 10)                110       
                                                                 
 dense_2 (Dense)             (None, 10)                110       
                                                                 
 dense_3 (Dense)             (None, 10)                110       
                                                                 
 dense_4 (Dense)             (None, 10)                110   

And save the models and training history - dump directory may need to be adjusted.

In [27]:
dump_dir = os.path.join("/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/operational_model/dump/", model_setting)
!mkdir -p $dump_dir

# save models to pickle
models = [model_1, model_2, model_3, model_4]

for model, i in zip(models, range(1,5)):
    save_to_pickle(model, f"{dump_dir}/model_{i}.pkl")

# save history to pickles
histories = [history_1, history_2, history_3, history_4]

for history, i in zip(histories, range(1,5)):
    save_to_pickle(history, f"{dump_dir}/history_{i}.pkl")


INFO:tensorflow:Assets written to: ram://c60c823e-af98-4fdb-80d3-7f5c869ae092/assets
INFO:tensorflow:Assets written to: ram://bb6994cd-f3f1-49a0-b915-5521f302f27a/assets
INFO:tensorflow:Assets written to: ram://42c15734-bcd2-4061-af68-76f77a10587c/assets
INFO:tensorflow:Assets written to: ram://a5bd9743-1b50-41d9-b1f6-eb0b348ae930/assets
INFO:tensorflow:Assets written to: ram://3462cb4f-0fd5-4192-a9ee-f94fd60aca5d/assets
INFO:tensorflow:Assets written to: ram://847c5fe5-228c-4ab5-8b1a-c3977c0717ab/assets
INFO:tensorflow:Assets written to: ram://a21fb0dc-54ad-4964-94f0-7e086a55abf0/assets
INFO:tensorflow:Assets written to: ram://efb65579-98c0-4b72-b518-48ad1a551acc/assets
